# Prediccion

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, roc_auc_score, f1_score, classification_report
)
import xgboost as xgb
import lightgbm as lgb
import json, time

In [2]:
# ── 1. CARGA ────────────────────────────────────────────────
df = pd.read_excel('Predios_Cercado_de_Lima_Ene-Mar_2025_4.xlsx')

# Renombrar columnas para facilitar manejo
df.columns = [
    'num_registro', 'fecha_adquisicion', 'fecha_declaracion', 'num_persona',
    'tipo_propietario', 'pct_propiedad', 'num_predio', 'cod_uso_predio',
    'uso_predio', 'area_terreno', 'area_comun_terreno', 'area_construida',
    'area_comun_construida', 'area_total_construida', 'pisos',
    'anio_construccion', 'mayor_anio_construccion', 'material_predio',
    'valor_terreno', 'valor_construccion_dep', 'valor_obras_comp',
    'autovaluo', 'afecto_imp_predial'
]

print(f"Filas originales: {len(df):,}")

Filas originales: 229,131


In [3]:
# ── 2. FEATURE ENGINEERING ──────────────────────────────────
# Limpiar año construccion (valores absurdos < 1800 → NaN)
df.loc[df['anio_construccion'] < 1800, 'anio_construccion'] = np.nan
df.loc[df['mayor_anio_construccion'] < 1800, 'mayor_anio_construccion'] = np.nan

# Edad del predio
df['edad_predio'] = 2025 - df['anio_construccion']  # NaN donde no hay año

# Ratio construido/terreno (densidad)
df['ratio_construido'] = np.where(
    df['area_terreno'] > 0,
    df['area_total_construida'] / df['area_terreno'],
    0
)

# Área total (terreno + construida)
df['area_suma'] = df['area_terreno'] + df['area_total_construida']

# Fechas → años
df['anio_adquisicion'] = pd.to_datetime(df['fecha_adquisicion'], errors='coerce').dt.year
df['antiguedad_propiedad'] = 2025 - df['anio_adquisicion']

# ── 2. FEATURE ENGINEERING ──────────────────────────────────
# ... (tu código existente, sin cambios) ...
df['area_comun_total'] = df['area_comun_terreno'] + df['area_comun_construida']

# ── 2.5 AGRUPACIÓN USO DE PREDIO ────────────────────────────
uso_map = {
    # VIVIENDA
    'Vivienda':                          'Vivienda',
    'Deposito de vivienda':              'Vivienda',
    'Vivienda colectiva - habitaciones': 'Vivienda',
    'Aires':                             'Vivienda',
    # COMERCIAL
    'Bodega':                            'Comercial',
    'Tienda por departamentos, supermercado, hiperbodega y similares': 'Comercial',
    'Puesto (o stand) en galería comercial':  'Comercial',
    'Puesto (o stand) en mercado o campo ferial': 'Comercial',
    'Galería comercial':                 'Comercial',
    'Mercado o campo ferial':            'Comercial',
    'Centro comercial':                  'Comercial',
    'Bazar y regalos':                   'Comercial',
    'Comercial - no identificado':       'Comercial',
    'Otros usos comerciales no especificados': 'Comercial',
    # SERVICIOS
    'Oficina':                           'Servicios',
    'Local de servicios (empresarial o profesional)': 'Servicios',
    'Estacionamiento de oficinas':       'Servicios',
    'Depósito de oficina':               'Servicios',
    'Agencia de entidad financiera':     'Servicios',
    'Sede de entidad financiera':        'Servicios',
    'Agencia de seguros o afp':          'Servicios',
    'Correo o teléfono':                 'Servicios',
    'Internet':                          'Servicios',
    'Centro de estética':                'Servicios',
    'Lavanderia de carros':              'Servicios',
    'Ventas de departamentos':           'Servicios',
    # INDUSTRIAL
    'Industria manufacturera':           'Industrial',
    'Taller':                            'Industrial',
    'Almacén o depósito':                'Industrial',
    'Tendal':                            'Industrial',
    'Chatarrerías':                      'Industrial',
    'Grifo venta de combustibles para vehiculos': 'Industrial',
    'Local venta de combustibles de uso doméstico': 'Industrial',
    'Distribución o transmisión de energía eléctrica': 'Industrial',
    'Sub-estación eléctrica o telefónica': 'Industrial',
    'Centro de captación, purificación y distribución de agua': 'Industrial',
    'Laboratorio de ensayos ambientales y similares': 'Industrial',
    # EDUCACIÓN
    'Educación inicial, primaria y/o secundaria': 'Educación',
    'Educación técnica y cenecapes':     'Educación',
    'Educación superior (universitaria o instituto)': 'Educación',
    'Educación pre universitaria (academia)': 'Educación',
    'Otros usos educativos nep':         'Educación',
    'Cuna':                              'Educación',
    # SALUD
    'Hospital o clínica':                'Salud',
    'Centro de salud':                   'Salud',
    'Centro veterinario':                'Salud',
    'Hospicio, albergue, puericultorio o asilo': 'Salud',
    'Centro de rehabilitación y asistencia social (c.r.a.s.-cárceles)': 'Salud',
    'Otros usos de salud nep':           'Salud',
    'Comedor popular':                   'Salud',
    'Centro de asistencia social':       'Salud',
    # RECREACIÓN
    'Restaurantes, bares y cantinas':    'Recreación',
    'Bingo, casino de juego, pinball, tragamonedas o similar': 'Recreación',
    'Club, círculo o centro deportivo':  'Recreación',
    'Club social':                       'Recreación',
    'Gimnasio y similares':              'Recreación',
    'Cine':                              'Recreación',
    'Teatro':                            'Recreación',
    'Cafe teatro, cabaret, centro nocturno, boite': 'Recreación',
    'Peña, discoteca, salón de bailes o similares': 'Recreación',
    'Museo':                             'Recreación',
    'Centro cultural':                   'Recreación',
    'Sala de exposición, galería de arte': 'Recreación',
    'Biblioteca':                        'Recreación',
    'Estadio, hipódromo o coliseo':      'Recreación',
    'Otros usos deportivos nep':         'Recreación',
    'Otros servicios de recreación o esparcimiento nep': 'Recreación',
    'Otros usos culturales nep':         'Recreación',
    'Radio o televisión':                'Recreación',
    'Periodismo':                        'Recreación',
    # TERRENO SIN CONSTRUIR
    'Terreno sin construir':             'Sin construir',
    'Cochera':                           'Sin construir',
    'Playa o edificio de estacionamiento': 'Sin construir',
}

df['uso_macro'] = df['uso_predio'].map(uso_map).fillna('Otros')
print(df['uso_macro'].value_counts())

# ── 3. DEFINIR FEATURES ──────────────────────────────────────
NUM_FEATURES = [
    'area_terreno', 'area_construida', 'area_total_construida',
    'area_comun_terreno', 'area_comun_construida', 'area_comun_total',
    'area_suma', 'pisos', 'edad_predio', 'ratio_construido',
    'pct_propiedad', 'antiguedad_propiedad'
]

CAT_FEATURES = [
    'uso_macro',          # ← reemplaza 'uso_predio'
    'material_predio',
    'tipo_propietario'
]

ALL_FEATURES = NUM_FEATURES + CAT_FEATURES



uso_macro
Vivienda         118645
Comercial         73309
Servicios         12285
Sin construir     11253
Industrial         8407
Recreación         3150
Otros              1200
Educación           727
Salud               155
Name: count, dtype: int64


In [4]:
# ── 4. PIPELINE DE PREPROCESAMIENTO ─────────────────────────
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, NUM_FEATURES),
    ('cat', cat_transformer, CAT_FEATURES)
])

print("Preprocesamiento configurado ✓")

Preprocesamiento configurado ✓


In [5]:
# ── 5. FUNCIÓN AUXILIAR DE MÉTRICAS ─────────────────────────
def reg_metrics(name, y_true, y_pred, y_pred_log=False, log_transform=False):
    """Si el modelo predijo en log, revertimos para métricas interpretables."""
    if log_transform:
        y_pred_orig = np.expm1(y_pred)
        y_true_orig = np.expm1(y_true)
    else:
        y_pred_orig = y_pred
        y_true_orig = y_true
    r2  = r2_score(y_true, y_pred)           # R² en el espacio del modelo
    mae = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse= np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    mape= np.mean(np.abs((y_true_orig - y_pred_orig) / (y_true_orig + 1))) * 100
    print(f"  {name}")
    print(f"    R²:   {r2:.4f}   |  MAE:  S/ {mae:,.0f}   |  RMSE: S/ {rmse:,.0f}   |  MAPE: {mape:.2f}%")
    return {'modelo': name, 'R2': round(r2,4), 'MAE': round(mae,2),
            'RMSE': round(rmse,2), 'MAPE': round(mape,2)}

def clf_metrics(name, y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    f1  = f1_score(y_true, y_pred)
    print(f"  {name}")
    print(f"    Accuracy: {acc:.4f}   |  AUC-ROC: {auc:.4f}   |  F1: {f1:.4f}")
    return {'modelo': name, 'Accuracy': round(acc,4), 'AUC_ROC': round(auc,4), 'F1': round(f1,4)}


In [6]:

# ── 6. ENTRENAMIENTO DE ALGORITMOS ──────────────────────

dfB = df[ALL_FEATURES + ['valor_terreno']].copy()
dfB = dfB[dfB['valor_terreno'] > 0].copy()
y_B = np.log1p(dfB['valor_terreno'])
X_B = dfB[ALL_FEATURES]

X_B_tr, X_B_te, y_B_tr, y_B_te = train_test_split(
    X_B, y_B, test_size=0.2, random_state=42)

ppB = ColumnTransformer([
    ('num', num_transformer, NUM_FEATURES),
    ('cat', cat_transformer, CAT_FEATURES)
])
X_B_tr_pp = ppB.fit_transform(X_B_tr)
X_B_te_pp = ppB.transform(X_B_te)
print(f"Train: {X_B_tr_pp.shape[0]:,} | Test: {X_B_te_pp.shape[0]:,}")

results_B = []

print("\nRandom Forest Regressor…")
t0 = time.time()
rfB = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=5,
                              n_jobs=-1, random_state=42)
rfB.fit(X_B_tr_pp, y_B_tr)
pred = rfB.predict(X_B_te_pp)
print(f"  Tiempo: {time.time()-t0:.1f}s")
results_B.append(reg_metrics("Random Forest", y_B_te, pred, log_transform=True))

print("\nXGBoost Regressor…")
t0 = time.time()
xgbB = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=7,
                          subsample=0.8, colsample_bytree=0.8,
                          n_jobs=-1, random_state=42, verbosity=0)
xgbB.fit(X_B_tr_pp, y_B_tr)
pred = xgbB.predict(X_B_te_pp)
print(f"  Tiempo: {time.time()-t0:.1f}s")
results_B.append(reg_metrics("XGBoost", y_B_te, pred, log_transform=True))

print("\nLightGBM Regressor…")
t0 = time.time()
lgbB = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=7,
                           num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                           n_jobs=-1, random_state=42, verbose=-1)
lgbB.fit(X_B_tr_pp, y_B_tr)
pred = lgbB.predict(X_B_te_pp)
print(f"  Tiempo: {time.time()-t0:.1f}s")
results_B.append(reg_metrics("LightGBM", y_B_te, pred, log_transform=True))


# ── 7. RESUMEN FINAL ─────────────────────────────────────────
print("\n" + "=" * 60)
print("  RESUMEN CONSOLIDADO")
print("=" * 60)


print("\n📊 BLOQUE B — Valor Terreno 2025 (Regresión)")
df_B = pd.DataFrame(results_B).set_index('modelo')
print(df_B.to_string())


# Guardar resultados para el reporte
import json
results_json = {

    'bloque_B': results_B,

}
with open('results.json', 'w') as f:
    json.dump(results_json, f, indent=2)

print("\n✅ Resultados guardados en results.json")

Train: 183,199 | Test: 45,800

Random Forest Regressor…
  Tiempo: 67.0s
  Random Forest
    R²:   0.9704   |  MAE:  S/ 11,780   |  RMSE: S/ 255,745   |  MAPE: 19.90%

XGBoost Regressor…
  Tiempo: 5.2s
  XGBoost
    R²:   0.9659   |  MAE:  S/ 15,895   |  RMSE: S/ 388,092   |  MAPE: 22.28%

LightGBM Regressor…
  Tiempo: 2.2s
  LightGBM
    R²:   0.9642   |  MAE:  S/ 15,936   |  RMSE: S/ 357,334   |  MAPE: 23.06%

  RESUMEN CONSOLIDADO

📊 BLOQUE B — Valor Terreno 2025 (Regresión)
                   R2       MAE       RMSE   MAPE
modelo                                           
Random Forest  0.9704  11780.49  255744.97  19.90
XGBoost        0.9659  15894.72  388092.29  22.28
LightGBM       0.9642  15935.88  357333.53  23.06

✅ Resultados guardados en results.json


In [7]:
# ── 8. EXPORTAR PARA STREAMLIT ───────────────────────────────
import joblib

# Seleccionar el mejor modelo por RMSE
df_B = pd.DataFrame(results_B).set_index('modelo')
mejor_nombre = df_B['RMSE'].idxmin()
print(f"\n🏆 Mejor modelo: {mejor_nombre}")

# Mapear nombre → modelo entrenado
modelos_dict = {
    "Random Forest": rfB,
    "XGBoost":       xgbB,
    "LightGBM":      lgbB,
}
mejor_modelo = modelos_dict[mejor_nombre]

# Guardar preprocesador + mejor modelo por separado
joblib.dump(ppB,          'preprocessor.pkl')
joblib.dump(mejor_modelo, 'model.pkl')

# Guardar metadata para la app
model_info = {
    'mejor_modelo':  mejor_nombre,
    'metricas':      df_B.loc[mejor_nombre].to_dict(),
    'num_features':  NUM_FEATURES,
    'cat_features':  CAT_FEATURES,
    'all_features':  ALL_FEATURES,
}
with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2, ensure_ascii=False)

print("✅ preprocessor.pkl, model.pkl y model_info.json guardados")


🏆 Mejor modelo: Random Forest
✅ preprocessor.pkl, model.pkl y model_info.json guardados
